#### CSCE 670 :: Information Storage & Retrieval :: Texas A&M University :: Spring 2025


# Homework 1:  OASIS Search Engine, The Beginning

### 100 points [4% of your final grade]

### Due: February 5 (Wednesday) by 11:59pm

*Goals of this homework:* In this homework you will get first hand experience building a text-based mini search engine + explore some LLM capabilities.

*Submission instructions (Canvas):* To submit your homework, rename this notebook as `UIN_hw1.ipynb`. For example, my homework submission would be something like `555001234_hw1.ipynb`. Submit this notebook via Canvas (look for the homework 1 assignment there). Your notebook should be completely self-contained, with the results visible in the notebook. We should not have to run any code from the command line, nor should we have to run your code within the notebook (though we reserve the right to do so). So please run all the cells for us, and then submit.

*Late submission policy:* For this homework, you may use as many late days as you like (up to the 5 total allotted to you).

*Collaboration policy:* You are expected to complete each homework independently. Your solution should be written by you without the direct aid or help of anyone else. However, we believe that collaboration and team work are important for facilitating learning, so we encourage you to discuss problems and general problem approaches (but not actual solutions) with your classmates. You may post on Canvas, search StackOverflow, even use ChatGPT. But if you do get help in this way, you must inform us by **filling out the Collaboration Declarations at the bottom of this notebook**. See the course syllabus for details.

*Example: I found helpful code on stackoverflow at https://stackoverflow.com/questions/11764539/writing-fizzbuzz that helped me solve Problem 2.*

The basic rule is that no student should explicitly share a solution with another student (and thereby circumvent the basic learning process), but it is okay to share general approaches, directions, and so on. If you feel like you have an issue that needs clarification, feel free to contact either me or the TA.

# Dataset: Enron Email Dataset

We are providing you with a small collection of emails from the Enron Email Dataset. The Enron Email Dataset was collected and prepared by the CALO Project (A Cognitive Assistant that Learns and Organizes). It contains data from about 150 users, mostly senior management of Enron. The full corpus contains a total of about 0.5M messages (https://www.cs.cmu.edu/~enron/). For this homework, we will use a small subset of the data. The subset contains 814 emails extracted from the `_sent_mail` of Arnold-J. We have zipped the 814 files (each file contains the information of an email). The zipped file is available on Canvas as `enron_814.zip`. The subset we provide is about 1.1MB. You should treat each email as a unique document to be indexed by your system. You can download the data from Canvas to your local filesystem. We're going to use these emails as the basis of OASIS, our Open Access Searchable Information System!


Below is an example of one email.

```text
Message-ID: <33025919.1075857594206.JavaMail.evans@thyme>
Date: Wed, 13 Dec 2000 13:09:00 -0800 (PST)
From: john.arnold@enron.com
To: slafontaine@globalp.com
Subject: re:spreads
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: John Arnold
X-To: slafontaine@globalp.com @ ENRON
X-cc:
X-bcc:
X-Folder: \John_Arnold_Dec2000\Notes Folders\'sent mail
X-Origin: Arnold-J
X-FileName: Jarnold.nsf

saw a lot of the bulls sell summer against length in front to mitigate
margins/absolute position limits/var.  as these guys are taking off the
front, they are also buying back summer.  el paso large buyer of next winter
today taking off spreads.  certainly a reason why the spreads were so strong
on the way up and such a piece now.   really the only one left with any risk
premium built in is h/j now.   it was trading equivalent of 180 on access,
down 40+ from this morning.  certainly if we are entering a period of bearish
to neutral trade, h/j will get whacked.  certainly understand the arguments
for h/j.  if h settles $20, that spread is probably worth $10.  H 20 call was
trading for 55 on monday.  today it was 10/17.  the market's view of
probability of h going crazy has certainly changed in past 48 hours and that
has to be reflected in h/j.




slafontaine@globalp.com on 12/13/2000 04:15:51 PM
To: slafontaine@globalp.com
cc: John.Arnold@enron.com
Subject: re:spreads



mkt getting a little more bearish the back of winter i think-if we get another
cold blast jan/feb mite move out. with oil moving down and march closer flat
px
wide to jan im not so bearish these sprds now-less bullish march april as
well.
```

# Part 1: Read and Parse the Email Data (20 points)

Recall how we handled file input in Homework 0? Well, here, our goal is to read the emails so that we can begin to tokenize them later. For this step, you should read the dataset and print the emails. Note that our dataset contains multiple files. You will need to write Python code to read from these files, and then build a list to store the documents. Each item in the list should be a dictionary containing the `Document-ID` as the key, and email content as the value. You can discard all the supplementary information of the email, e.g., `Date`, `From`, `To`, `Subject`, etc.

A document should look like:

```text
{'Document-ID': '33025919.1075857594206',
'content': 'saw a lot of the bulls sell summer against length in front to mitigate
margins/absolute position limits/var.  as these guys are taking off the
front, they are also buying back summer.  el paso large buyer of next winter
today taking off spreads.  certainly a reason why the spreads were so strong
on the way up and such a piece now.   really the only one left with any risk
premium built in is h/j now.   it was trading equivalent of 180 on access,
down 40+ from this morning.  certainly if we are entering a period of bearish
to neutral trade, h/j will get whacked.  certainly understand the arguments
for h/j.  if h settles $20, that spread is probably worth $10.  H 20 call was
trading for 55 on monday.  today it was 10/17.  the market's view of
probability of h going crazy has certainly changed in past 48 hours and that
has to be reflected in h/j.




slafontaine@globalp.com on 12/13/2000 04:15:51 PM
To: slafontaine@globalp.com
cc: John.Arnold@enron.com
Subject: re:spreads



mkt getting a little more bearish the back of winter i think-if we get another
cold blast jan/feb mite move out. with oil moving down and march closer flat
px
wide to jan im not so bearish these sprds now-less bullish march april as
well.'
}
```

For this homework, you should treat the email content as a document and the Message-ID as the document ID.

## Print the first two documents (5 points)

Your output should look like this:

DocumentID Document

33025919.1075857594206 saw a lot of the bulls sell summer ......

...


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# your code here
# please print out the first 2 docs
import os
import re

folder_path="/content/drive/My Drive/Masters Spring 2025/Information Storage and Retrieval/enron_814"
documents =[]
files =sorted(os.listdir(folder_path), key=lambda x: int(x.rstrip("_")))
actual_content = []

def parse_body(content):
    body=re.search(r"(\r?\n){2,}",content)
    if body:
      return content[body.end():]
    return content

def parse_message_id(content):
      match =re.search(r"Message-ID: <(\d+\.\d+)",content)
      if match:
         return match.group(1)
      return "Unknown ID"

for file_name in files:
     file_path= os.path.join(folder_path,file_name)
     with open(file_path,"r", encoding="utf-8") as file:
          content = file.read()

     body = parse_body(content)
     message_id = parse_message_id(content)
     documents.append({message_id : body.strip().replace("\n"," ")})
     actual_content.append({message_id:body})



print(documents[:2])

[{'33025919.1075857594206': "saw a lot of the bulls sell summer against length in front to mitigate  margins/absolute position limits/var.  as these guys are taking off the  front, they are also buying back summer.  el paso large buyer of next winter  today taking off spreads.  certainly a reason why the spreads were so strong  on the way up and such a piece now.   really the only one left with any risk  premium built in is h/j now.   it was trading equivalent of 180 on access,  down 40+ from this morning.  certainly if we are entering a period of bearish  to neutral trade, h/j will get whacked.  certainly understand the arguments  for h/j.  if h settles $20, that spread is probably worth $10.  H 20 call was  trading for 55 on monday.  today it was 10/17.  the market's view of  probability of h going crazy has certainly changed in past 48 hours and that  has to be reflected in h/j.     slafontaine@globalp.com on 12/13/2000 04:15:51 PM To: slafontaine@globalp.com cc: John.Arnold@enron.c

Now that you can read the documents, let's move on to tokenization. We are going to simplify things for you:
1. You should **lowercase** all words.
2. Replace line breaks (e.g., \n, \n\n), punctuations, dashes and splash (e.g., -, /) and special characters (\u2019, \u2005, etc.) with empty space (" ").
3. Tokenize the documents by splitting on whitespace.
4. Then only keep words that have a-zA-Z in them.

In [3]:
# your code here
import string
import re

for doc in documents:
    for key,value in doc.items():

          processed_text = value.lower()
          processed_text = processed_text.replace("\n"," ")   #Replacing line breaks with whitespaces
          processed_text =processed_text.replace("-", " ")    #Replacing - (dashes) with whitespaces
          processed_text= processed_text.replace("/", " ")    #Replacing / (slashe) with whitespaces
          translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))  # Replacing punctuation marks with whitespace
          processed_text =processed_text.translate(translator)
          processed_text = re.sub(r'[^a-zA-Z0-9 \n\.]', '', processed_text) #Replacing special characters with white space
          tokens= processed_text.split()   #Tokenising the strings
          filtered_tokens =[item for item in tokens if re.match('^[a-zA-Z]+$', item)]

          doc[key]=filtered_tokens


## Print the first two documents after tokenizing (5 points)

Once you have your parser working, you should print the first two documents (documentID and tokens).

Your output should look like this:

* DocumentID Tokens



In [4]:
# your code and output here
print(documents[:2])

[{'33025919.1075857594206': ['saw', 'a', 'lot', 'of', 'the', 'bulls', 'sell', 'summer', 'against', 'length', 'in', 'front', 'to', 'mitigate', 'margins', 'absolute', 'position', 'limits', 'var', 'as', 'these', 'guys', 'are', 'taking', 'off', 'the', 'front', 'they', 'are', 'also', 'buying', 'back', 'summer', 'el', 'paso', 'large', 'buyer', 'of', 'next', 'winter', 'today', 'taking', 'off', 'spreads', 'certainly', 'a', 'reason', 'why', 'the', 'spreads', 'were', 'so', 'strong', 'on', 'the', 'way', 'up', 'and', 'such', 'a', 'piece', 'now', 'really', 'the', 'only', 'one', 'left', 'with', 'any', 'risk', 'premium', 'built', 'in', 'is', 'h', 'j', 'now', 'it', 'was', 'trading', 'equivalent', 'of', 'on', 'access', 'down', 'from', 'this', 'morning', 'certainly', 'if', 'we', 'are', 'entering', 'a', 'period', 'of', 'bearish', 'to', 'neutral', 'trade', 'h', 'j', 'will', 'get', 'whacked', 'certainly', 'understand', 'the', 'arguments', 'for', 'h', 'j', 'if', 'h', 'settles', 'that', 'spread', 'is', 'prob

## Dictionary Size (5 points)

Next you should report the size of your dictionary, that is, how many unique tokens among all the documents.




In [5]:
# your code and output here
unique = set()
for i in documents:
  for value in i.values():
     unique.update(value)
print(len(unique))

9343


## Top-20 Words (5 points)

Finally, you should print a list of the top-20 most popular words by counting among all documents.


Your output should look like this:

* Rank. Token, Count

1. awesome, 20
2. cool, 15
3. ...

In [6]:
from collections import Counter

# your code and output here
words_counts ={}
for doc in documents:
      for tokens in doc.values():
        for token in tokens:
              if token in words_counts:
                words_counts[token] += 1
              else:
                words_counts[token] = 1

no_of_words = Counter(words_counts)
top_20_words = no_of_words.most_common(20)
i=1
print("Rank. Token Count")
for l, m in top_20_words:
    print(f"{i}.   {l}, {m}")
    i+=1

Rank. Token Count
1.   the, 4111
2.   to, 3768
3.   ect, 2600
4.   enron, 2293
5.   a, 1879
6.   and, 1795
7.   you, 1749
8.   of, 1716
9.   i, 1694
10.   on, 1590
11.   john, 1514
12.   in, 1465
13.   hou, 1345
14.   com, 1317
15.   is, 1202
16.   for, 1165
17.   arnold, 999
18.   subject, 936
19.   s, 869
20.   it, 865


# Part 2: Boolean Retrieval (30 points)

In this part you will build an inverted index to support Boolean retrieval. You should use the tokenization strategy from above.

We require your index to support AND, OR, NOT queries.

Search for the queries below using your index and print out matching documents (for each query, print out 5 matching documents):
* buyer
* margins AND limits
* winter OR summer
* buyers AND risk AND NOT crazy
* never OR know

Recall, that you should apply the exact same pre-processing strategies to the query as we do to the documents.

The output should like this:
* DocumentID Document

To make our life easier, please output the DocumentIDs in lexicographic order.

In [7]:
# build the index here
# add cells as needed to organize your code
import re
from collections import defaultdict

queries =[    "buyer",
    "margins AND limits",
    "winter OR summer",
    "buyers AND risk AND NOT crazy",
    "never OR know" ]

def preprocessing(text):
    text =text.lower()
    text = text.replace("\n", " ")
    text =text.replace("-", " ")
    text= text.replace("/", " ")
    translator =str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    text =text.translate(translator)
    text= re.sub(r'[^a-zA-Z0-9 \n\.]','', text)
    text = re.sub(r'[^\w\s]', '',text)
    return text.split()


preprocessed_queries =[]

for i in queries:
      preprocessed_query = preprocessing(i)
      preprocessed_queries.append(preprocessed_query)


inverted_index = defaultdict(set)
full_document ={}

for document in documents:
    for doc_id,doc in document.items():
         full_document[doc_id] = " ".join(doc)
         for i in set(doc):
              inverted_index[i].add(doc_id)

In [21]:
def boolean_retrieval(query):

    new_set =set()
    operation = None
    for term in query:
        if term in {"and", "or", "not","AND","OR","NOT"}:
            operation =term.upper()
        else:
            doc_set = inverted_index.get(term, set())
            if operation == "AND":
                new_set &= doc_set
            elif operation == "OR":
                new_set |= doc_set
            elif operation == "NOT":
                new_set -= doc_set
            else:
                new_set = doc_set

    return sorted(new_set)[:5]

for i in preprocessed_queries:
    results =boolean_retrieval(i)
    print(f"\nQuery: {i}")
    for j in results:
        print(f"{j} {full_document[j]}")


Query: ['buyer']
13960264.1075857658698 very useful thx keep me posted caroline abramo enron am to john arnold hou ect ect mike maggi corp enron enron jennifer fraser hou ect ect cc per sekse ny ect ect subject fund views hi all the funds are trying to figure out what the play is for next year major divergence of opinions most everyone we talk to takes a macro view dwight anderson from tudor thinks anything above is a sale from the perspective of shut in industrial demand he believes that between no industrial basic industry type can operate he tracks all the plant closures similar to what elena does in mike robert s group but it seems on a more comprehensive level jen it would be good for fundamentals to track this number do some scenario analysis on it under various economic conditions like recession i will try to find out what his total number for turned back gas is just ammonia is a little more than bcf which does not seem all that meaningful but the total may bring us back into b

## Running the five queries (4 points each, 20 points in total)

In [9]:
# search for the input using your index and print out ids of matching documents.
for i in preprocessed_queries:
    results =boolean_retrieval(i)
    print(f"\nQuery: {i}")
    for j in results:
        print(f"{j}")


Query: ['buyer']
13960264.1075857658698
17195279.1075857655281
2268604.1075857652949
25732708.1075857656969
28376645.1075857655238

Query: ['margins', 'and', 'limits']
33025919.1075857594206

Query: ['winter', 'or', 'summer']
10353423.1075857652669
10537445.1075857655390
10603457.1075857597605
11971129.1075857654956
12541953.1075857657123

Query: ['buyers', 'and', 'risk', 'and', 'not', 'crazy']
2726985.1075857655016

Query: ['never', 'or', 'know']
10008095.1075857595829
10216181.1075857658324
10337786.1075857649963
10353423.1075857652669
10396784.1075857651750


Now show the results for the query: `buyer`

In [10]:
tokens = preprocessing("buyer")
results = boolean_retrieval(tokens)
print(f"\nQuery: {tokens}")

for message_id in results:
    found = False  # Track if the message_id is found
    for doc in actual_content:
        if message_id in doc:
            print(f"{message_id}\n{doc[message_id]}")  # Print with preserved format
            found = True
            break  # Stop searching once found
    if not found:
        print(f"Warning: message_id '{message_id}' not found in actual_content")



Query: ['buyer']
13960264.1075857658698
very useful...thx.   keep me posted




Caroline Abramo@ENRON
12/22/2000 11:41 AM
To: John Arnold/HOU/ECT@ECT, Mike Maggi/Corp/Enron@Enron, Jennifer 
Fraser/HOU/ECT@ECT
cc: Per Sekse/NY/ECT@ECT 
Subject: fund views

Hi- all the funds are trying to figure out what the play is for next year- 
major divergence of opinions.  Most everyone we talk to takes a macro view.

+ Dwight Anderson from Tudor thinks anything above $6 is a sale from the 
perspective of shut in industrial demand- he believes that between $6-7 no 
industrial (basic industry type) can operate.  He tracks all the plant 
closures similar to what Elena does in Mike Robert's group but it seems on a 
more comprehensive level (Jen- it would be good for fundamentals to track 
this number- do some scenario analysis on it under various economic 
conditions- like recession!!).  I will try to find out what his total number 
for turned back gas is- just ammonia is a little more than 1/2 Bcf w

*Now* show the results for the query: `margins AND limits`

---




In [ ]:
# your code here
tokens = preprocessing("margins AND limits")
results = boolean_retrieval(tokens)
print(f"\nQuery: {tokens}")
for j in results:
    for doc in actual_content:
      if j in doc:
       print(f"{j}\n{doc[j]}")


Query: ['margins', 'and', 'limits']
33025919.1075857594206
saw a lot of the bulls sell summer against length in front to mitigate 
margins/absolute position limits/var.  as these guys are taking off the 
front, they are also buying back summer.  el paso large buyer of next winter 
today taking off spreads.  certainly a reason why the spreads were so strong 
on the way up and such a piece now.   really the only one left with any risk 
premium built in is h/j now.   it was trading equivalent of 180 on access, 
down 40+ from this morning.  certainly if we are entering a period of bearish 
to neutral trade, h/j will get whacked.  certainly understand the arguments 
for h/j.  if h settles $20, that spread is probably worth $10.  H 20 call was 
trading for 55 on monday.  today it was 10/17.  the market's view of 
probability of h going crazy has certainly changed in past 48 hours and that 
has to be reflected in h/j.




slafontaine@globalp.com on 12/13/2000 04:15:51 PM
To: slafontaine@glob

Now show the results for the query: `winter OR summer`

In [ ]:
# your code here
tokens = preprocessing("winter OR summer")
results = boolean_retrieval(tokens)
print(f"\nQuery: {tokens}")
for j in results:
    for doc in actual_content:
      if j in doc:
       print(f"{j}\n{doc[j]}")


Query: ['winter', 'or', 'summer']
10353423.1075857652669
maybe.  hydro situation dire in west.  think water levels are at recent 
historical lows.  problem is from gas standpoint, west is an island right 
now.  every molecle that can go there is.  so will provide limited support to 
prices in east.  hydro in east is actually very healthy.  would assume your 
markets are targeting eastern u.s. so i dont know if hydro problem in west is 
that relevant.




Sarah Mulholland
04/04/2001 08:09 AM
To: John Arnold/HOU/ECT@ECT
cc:  
Subject: Re: us fuel 4/2/01

interesting comment from singapore........

hope things are going well up there.
---------------------- Forwarded by Sarah Mulholland/HOU/ECT on 04/04/2001 
08:08 AM ---------------------------


Hans Wong
04/04/2001 08:05 AM
To: Sarah Mulholland/HOU/ECT@ECT
cc: Niamh Clarke/LON/ECT@ECT, Stewart Peter/LON/ECT@ECT, Caroline 
Cronin/EU/Enron@Enron, Angela Saenz/ENRON@enronXgate 
Subject: Re: us fuel 4/2/01  

i was reading something inter

Now show the results for the query: `buyers AND risk AND NOT crazy`

In [ ]:
# your code here
tokens = preprocessing("buyers AND risk AND NOT crazy")
results = boolean_retrieval(tokens)
print(f"\nQuery: {tokens}")
for j in results:
    for doc in actual_content:
      if j in doc:
       print(f"{j}\n{doc[j]}")


Query: ['buyers', 'and', 'risk', 'and', 'not', 'crazy']
2726985.1075857655016
---------------------- Forwarded by John Arnold/HOU/ECT on 03/06/2001 07:48 
AM ---------------------------
From: Ann M Schmidt@ENRON on 03/05/2001 08:23 AM
To: Ann M Schmidt/Corp/Enron@ENRON
cc:  (bcc: John Arnold/HOU/ECT)
Subject: Enron Mentions - 03-04-01

Utility Deregulation: Square Peg, Round Hole?
The New York Times, 03/04/01

3 Executives Considered to Head Military
Los Angeles Times, 03/04/01

Bush leaning toward execs for military
The Seattle Times, 03/04/01

Enron's Chief Denies Role as Energy Villain / Critics regard Kenneth Lay as 
deregulation opportunist
The San Francisco Chronicle, 03/04/01

Enron boss says he's not to blame for profits in energy crisis
Associated Press Newswires, 03/04/01

The Stadium Curse? / Some stocks swoon after arena deals
The San Francisco Chronicle, 03/04/01



Money and Business/Financial Desk; Section 3
ECONOMIC VIEW
Utility Deregulation: Square Peg, Round Hole?
By

Now show the results for the query: `never OR know`

In [ ]:
# your code here
tokens = preprocessing("never OR know")
results = boolean_retrieval(tokens)
print(f"\nQuery: {tokens}")
for j in results:
    for doc in actual_content:
      if j in doc:
       print(f"{j}\n{doc[j]}")


Query: ['never', 'or', 'know']
10008095.1075857595829
not me




Brian Hoskins@ENRON COMMUNICATIONS
10/23/2000 03:47 PM
To: John Arnold/HOU/ECT@ECT
cc:  
Subject: Re:

Never mind.  Did you do that?


Brian T. Hoskins
Enron Broadband Services
713-853-0380 (office)
713-412-3667 (mobile)
713-646-5745 (fax)
Brian_Hoskins@enron.net


----- Forwarded by Brian Hoskins/Enron Communications on 10/23/00 03:54 PM 
-----

	John Cheng@ENRON
	Sent by: John Cheng@ENRON
	10/23/00 03:47 PM
		
		 To: John Cheng/NA/Enron@Enron
		 cc: Brian Hoskins/Enron Communications@ENRON COMMUNICATIONS, John 
Cheng/NA/Enron@Enron@ENRON COMMUNICATIONS, Fangming 
Zhu/Corp/Enron@ENRON@ENRON COMMUNICATIONS, Pablo 
Torres/Corp/Enron@ENRON@ENRON COMMUNICATIONS, Don Adam/Corp/Enron@ENRON@ENRON 
COMMUNICATIONS, Marlin Gubser/HOU/ECT@ECT, Mark Hall/HOU/ECT
		 Subject: Re:

All,

Sorry for the lengthy thread.  A service request has been opened with 
Microsoft on the IE5 issues and her application.

Regards,

-jkc



John Cheng

## Observations (10 points)
Does your boolean search engine find relevant documents for these queries? As in, would our customers be happy if we shipped this retrieval engine? Why or why not?

What is the impact of the pre-processing options? Do they impact your search quality?

*your discussion here*

Yes, my boolean search engine does find the relevant documents for all the queries. And I have printed them in the original format of the emails, as ideally your search engine should return them in the way they are, whenever our customers would search for particular query. I don't think the customers would be totally satisfied by our search engine, as although it gives us the top 5 documents, it does not rank them, the order/relevance of these documents is also important. The customers should get the most relevant documents at the top. The impact of pre-processing is that all the tokens (content of the email) as well as the tokens of the query , are brought down to a form that they are normalized, which increases the token matching efficiency, required for the boolean search, thus giving better search results.

# Part 3: Ranking (35 points)

For the third part, you will add ranking to your search system. Given a search query, our goal is to retrieve the top-5 most relevant emails by assigning a score to each document.

We will explore two ranking methods, each offering a different approach to scoring and ranking documents:


### Ranking method A: Ranking with vector space model with TF-IDF (15 points)

**Cosine:** You should use cosine as your scoring function.

**TFIDF:** For the **document vectors**, use the standard TF-IDF scores introduced in class. For the **query vector**, use **simple weights (the raw term frequency)**. For example:
* query: never $\rightarrow$ (1)
* query: never know $\rightarrow$ (1, 1)

**Query:**  `trade`

**Output:**
You should output the top-5 results plus the cosine score of each of these documents.  

The output should be like this:

Rank Scores DocumentID Document

---

You can additionally assume that your queries will contain at most three words. Be sure to normalize your vectors as part of the cosine calculation!


In [20]:
import numpy as np
from collections import Counter
from math import log, sqrt

def calculate_tf(doc_toks):
      counts_of_terms= Counter(doc_toks)
      total_no_of_terms =len(doc_toks)
      return {term: count /total_no_of_terms for term,count in counts_of_terms.items()}

def calculate_idf(docs):
    num_of_docs= len(docs)
    idf_vals ={}
    all_terms = set(term for doc in docs for term in doc)

    for term in all_terms:
        doc_freq = sum(1 for doc in docs if term in doc)
        idf_vals[term] = log(num_of_docs / (1 + doc_freq))

    return idf_vals

def compute_tfidf(doc_toks, idf_vals):
    tf_vals = calculate_tf(doc_toks)
    return {term: tf_vals[term] * idf_vals[term] for term in doc_toks if term in idf_vals}

def cosine_similarity(vec1,vec2):
        dot_products = sum(vec1[term] * vec2.get(term, 0) for term in vec1)
        norm1 =sqrt(sum(count**2 for count in vec1.values()))
        norm2= sqrt(sum(count**2 for count in vec2.values()))

        if norm1 > 0:
            vec1 ={k: v / norm1 for k, v in vec1.items()}
        if norm2 > 0:
             vec2= {k: v / norm2 for k, v in vec2.items()}

        return dot_products /(norm1 * norm2) if norm1 and norm2 else 0

doc_content =[list(j.values())[0] for j in documents]
doc_ids=[list(j.keys())[0] for j in documents]

idf_vals =calculate_idf(doc_content)
doc_tfidf_vectors =[compute_tfidf(doc, idf_vals) for doc in doc_content]

def process_query(query):
    query_tokens= query.lower().split()
    return compute_tfidf(query_tokens, idf_vals)

def search_query(query):
    query_vec = process_query(query)
    scores = [cosine_similarity(query_vec, doc_vect) for doc_vect in doc_tfidf_vectors]

    ranked_docs = sorted(zip(doc_ids, doc_content, scores),reverse=True,key=lambda x: x[2])[:5]

    print(f"\nQuery: {query}")
    print("Rank | Score  | DocumentID              | Document")

    for rank, (doc_id, _, score) in enumerate(ranked_docs, 1):
              preserved_text = None

              for doc in actual_content:
                  if doc_id in doc:
                      preserved_text = doc[doc_id]
                      break

              if preserved_text:
                  print(f"{rank}   | {score:.4f} | {doc_id}  | {preserved_text}")
                  print()
              else:
                  print(f"{rank}   | {score:.4f} | {doc_id}  | [Document Not Found]")

search_query("trade")


Query: trade
Rank | Score  | DocumentID              | Document
1   | 0.3223 | 32835197.1075857597302  | Hey:
I just want to confirm the trades I have in your book.
Trade #1.  I sell 4000 X @ 4652

Trade #2. I buy 4000 X @ 4652
  I sell 4000 X @ 4902

Trade #3 I buy 4000 X @ 5000
  I sell 4000 F @ 5000


Net result: I have 4000 F in your book @ 4902.
Thanks, 
John

2   | 0.3127 | 15827855.1075857658654  | torrey:
please set me up to trade crude.
John

3   | 0.2321 | 2752057.1075857658632  | Torrey:
Can you also approve Mike Maggi to trade crude as well.  Thanks for your help.
John

4   | 0.2306 | 3383202.1075857656796  | you fucker that's my trade.   i was trying to buy nines the last 20 minutes.  
all i got was scraps.  50-100.  i think it's a great trade.




slafontaine@globalp.com on 02/07/2001 01:41:44 PM
To: John.Arnold@enron.com
cc:  
Subject: Re: weather pop



that is nuts-good sale-im gonna sell jun or july otm calls at some point





5   | 0.1940 | 5340834.1075857658345  |

### Ranking method B: Ranking with BM25 (15 points)
Finally, let's try the BM25 approach for ranking. Refer to https://en.wikipedia.org/wiki/Okapi_BM25 for the specific formula. You could choose k_1 = 1.2 and b = 0.75 but feel free to try other options.

**Query:**  `gas floor`

**Output:**
You should output the top-5 results plus the BM25 score of each of these documents.  

The output should be like this:

Rank Scores DocumentID Document

In [15]:
# your code here
import numpy as np
import math

doc_texts = {list(doc.keys())[0]: list(doc.values())[0] for doc in documents}
doc_ids =list(doc_texts.keys())


k1 =1.2
b= 0.75


tf = {}
doc_lengths = {}
for doc_id,words in doc_texts.items():
        doc_lengths[doc_id] =len(words)
        tf[doc_id]= {}
        for word in words:
            tf[doc_id][word] =tf[doc_id].get(word, 0) + 1


df ={}
for doc_id, word_counts in tf.items():
    for word in word_counts:
        df[word] = df.get(word, 0) + 1

N = len(doc_texts)
idf = {word: math.log((N - df[word] + 0.5) / (df[word] + 0.5) + 1) for word in df}
avg_doc_length = np.mean(list(doc_lengths.values()))


def bm25(query, doc_id):
    score = 0
    for word in query:
        if word in tf[doc_id]:
            f = tf[doc_id][word]
            numer = idf[word] * (f * (k1 + 1))
            denom = f + k1 * (1 - b + b * (doc_lengths[doc_id] / avg_doc_length))
            score += numer / denom
    return score


query = "gas floor"
tokens = preprocessing(query)
scores = {doc_id: bm25(tokens,doc_id) for doc_id in doc_texts}
sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]


print("\nRank  Score      DocumentID  Document")

for rank, (doc_id, score) in enumerate(sorted_docs, start=1):
    preserved_text = None


    for doc in actual_content:
          if doc_id in doc:
              preserved_text = doc[doc_id]
              break

    if preserved_text:
         print(f"{rank:<5} | {score:.4f} | {doc_id:<10}| {preserved_text}")
         print()
    else:
        print(f"{rank:<5} | {score:.4f} | {doc_id:<10}| [Document Not Found]")

# print out the top-5 retrieved emails


Rank  Score      DocumentID  Document
1     | 7.6517 | 29559946.1075857598198| Thx for the spreadsheet.   2 questions : What time frame does this entail and 
does the correlation between the trader and AGG GAS include that trader's 
contribution to the floor's P&L.  In other words, is my P&L correlated with 
the floor or is it correlated to the rest of the floor absent me?


   
	Enron North America Corp.
	
	From:  Frank Hayden @ ENRON                           09/19/2000 02:41 PM
	

To: John Arnold/HOU/ECT@ECT
cc:  
Subject: 





2     | 6.8416 | 32732331.1075857597410| John:
I have asked Mike and Larry to spend half an hour each talking to you about 
opportunities on the gas floor.  Please advise if the following schedule is 
unacceptable.  I will be leaving today at 2:15.
Larry 4:00-4:30
Mike 4:30-5:00

Thanks,
John

3     | 6.8158 | 23846275.1075857658302| John:
I would like for you to come talk to a couple more people on the gas floor 
about a possible position down the road.  M

## Observations (5 points)
What do you observe? Are there key differences between the two ranking approaches? Briefly discuss in bullet points.


* Your observations:
   It seems that BM25 gives more contextually relevant documents, whereas tf-idf gives those documents where the terms are simply occuring. Therefore, BM25 search is better.
   In BM25, the two tokens 'gas' and 'floor' were not necessarily together in all the documents.
* Differences:
   Since BM25 considers the document length normalization, it is more like a probabilitic relevance model. Therefore, it gives more optimal results. Whereas tf-idf is more linear kind of model, hence it ranks those documents based on the frequency but does not normalize it, meaning longer documents with more words will get ranked higher.

   BM25 applies a saturation effect, something like a logarithmic graph if you plot the score vs frequency. It means more occuring words will not contribute much to the score after a point.
   Whereas, TF-IDF scores linearly as the frequency increases.

# Part 4: Cool LLM RAG Extension (15 points)

Finally, we give you an opportunity to explore using OASIS for Retrieval-Augmented Generation (RAG) with LLMs.
Here, the task is to retrieve the top-5 emails for a query you like. You will then pass the retrieved email content along with your question to the LLM and let it answer your question. Specifically, we want you:

* Query the LLM directly with your question.
* Retrieve the top-5 emails based on your query and pass them to the LLM along with your question.
* How is the RAG output different from the non-RAG output? Does retrieval help the LLM better answer your question?

We recommend using Gemini following the [instructions](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Prompting.ipynb) here.

Hint: Take a close look at the dataset and pick a specific, relevant query where retrieval can enhance the LLM’s response.

*What You Will Submit*
- Your query.
- The top-5 retrieved emails (including Document ID and ranking score).
- The LLM's response without retrieval.
- The LLM's response with retrieval (RAG).
- A brief analysis comparing both outputs.

We will grade this last part according to correctness, effort, and creativity.

## Step 1: Query the LLM Directly (Without Retrieval) (5 points)

In [16]:
!pip install -U -q "google-generativeai>=0.7.2" # Install the Python SDK

In [17]:
# your code here to prompt LLM
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [18]:
model = genai.GenerativeModel('gemini-2.0-flash')
response = model.generate_content("How much time did employee of Enron John ask Mike and Larry to spend on talking about opportunities on the gas floor?")
print(response.text)

I do not have access to specific internal communications from Enron or information about the specific conversations you mentioned between John, Mike, and Larry regarding gas floor opportunities.



## Step 2: Query the LLM with Retrieved Emails (RAG) (5 points)

In [19]:
# print out the LLM response without the email content
prompt = "How much time did John ask Mike and Larry to spend on talking about opportunities on the gas floor?"
print("Query-" ,prompt)
print()
print("Response after Querying the LLM Directly (Without Retrieval)-")
print(response.text)
print()

# print out the LLM response with the email content (RAG)
print("The top-5 retrieved emails")
corpus = " "
tokens = preprocessing(prompt)
scores = {doc_id: bm25(tokens,doc_id) for doc_id in doc_texts}
sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]


print("\nRank  Score      DocumentID  Document")

for rank, (doc_id, score) in enumerate(sorted_docs, start=1):
    preserved_text = None


    for doc in actual_content:
          if doc_id in doc:
              preserved_text = doc[doc_id]
              break

    if preserved_text:
         print(f"{rank:<5} | {score:.4f} | {doc_id:<10}| {preserved_text}")
         corpus += preserved_text
         print()
    else:
        print(f"{rank:<5} | {score:.4f} | {doc_id:<10}| [Document Not Found]")

print()
response2 = model.generate_content([prompt, corpus])
print("Response after Querying the LLM with Retrieved Emails (RAG)-")
print(response2.text)

Query- How much time did John ask Mike and Larry to spend on talking about opportunities on the gas floor?

Response after Querying the LLM Directly (Without Retrieval)-
I do not have access to specific internal communications from Enron or information about the specific conversations you mentioned between John, Mike, and Larry regarding gas floor opportunities.


The top-5 retrieved emails

Rank  Score      DocumentID  Document
1     | 42.1198 | 32732331.1075857597410| John:
I have asked Mike and Larry to spend half an hour each talking to you about 
opportunities on the gas floor.  Please advise if the following schedule is 
unacceptable.  I will be leaving today at 2:15.
Larry 4:00-4:30
Mike 4:30-5:00

Thanks,
John

2     | 16.9010 | 25124831.1075857599419| How about Tuesday at either 6:45 am or 2:15 pm my time.




David Redmond
08/25/2000 07:26 AM
To: John Arnold/HOU/ECT@ECT, Mike Maggi/Corp/Enron@Enron, John 
Disturnal/CAL/ECT@ECT
cc: Richard Lewis/LON/ECT@ECT, Peter Crilly/LON/E

## Discussion (5 points)

In this section, reflect on the performance of different ranking methods and the impact of retrieval on LLM responses. Consider:

- How did retrieval affect the LLM’s response? Did it improve factual accuracy or relevance?
- Were there cases where retrieval hurt performance (e.g., irrelevant documents, redundancy)?
- Any ideas for improving the ranking or retrieval process?

Keep your discussion *concise* and *insightful*, focusing on key takeaways from your experiments. Please answer in bullet points.

*your discussion here*

* Because of Retrieval, the LLM was able to provide me exact answer to what I was looking for, as it had the necessary information supplemented to specifically look up to for answering my query. Otherwise, only LLM could not answer my query as it asked for more data. It improved the factual accuracy, as it had more relevant data to refer to.
* I did not see any case where retrieval gave me worse answer than just the LLM, as per my queries which were relevant to the documents, it gave me the correct answers from the documents.
* What if we consider the time-stamps of the emails as a factor for ranking? Or maybe, also consider the designations of the employees (if we had to make this Search Engine specific to Enron)?
* Reflecting on the ranking methods performance, BM25 gave more contextually relevant documents than compared to TF-IDF as it gave those documents where simply words occured more.

# Collaboration Declarations

*You should fill out your collaboration declarations here.*
https://www.geeksforgeeks.org/python-remove-punctuation-from-string/
Referred this for learning how to remove punctuation and replace with spaces

https://stackoverflow.com/questions/23996118/replace-special-characters-in-a-string-in-python
Referred this for learning how to replace special characters with white space

Used GPT for Regex.

Referred Youtube to understand the difference in working between TF-IDF and BM25.  (https://www.youtube.com/watch?v=YL-3G5-xVYU)

Asked GPT to debug my code when developing the logic for tf-idf and bm25.

Referred to the github repository for the gemini model implementation for RAG.

Discussed with peers what outputs they were getting to ensure I was on the right track.

Asked a question anonymously on Canvas, to clear one of the doubt I had.
